# Variabilidade, intervalos e testes de hipótese

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_05/02_incerteza_e_testes_de_hipotese.ipynb)

O Notebook 01 estimou uma diferença observada. Agora perguntamos que variação
esperaríamos se o procedimento de obtenção dos dados pudesse ser repetido.

## 1. Antes da inferência: de onde vêm os dados?

Variabilidade amostral é a variação de uma estatística entre amostras obtidas
pelo mesmo procedimento. Ela não é sinônimo de erro de medição, transformação,
lacuna de cobertura ou mudança histórica.

| Situação | O intervalo amostral resolve? | Razão |
|---|---|---|
| amostra probabilística de uma população definida | pode quantificar parte da incerteza | o desenho sustenta a generalização |
| corpus completo para a pergunta | geralmente não é a questão central | a diferença do corpus já é observada |
| coleção de conveniência | não corrige a seleção | a reamostragem herda a composição disponível |

Os 24 registros fictícios serão tratados como se viessem de um procedimento
repetível apenas para compreender o método. Não faremos inferências históricas.

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_05'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Definido o alvo hipotético, podemos observar como uma estimativa varia entre
reamostras.

## 2. *Bootstrap* e distribuição da estimativa

![Uma amostra observada produz três reamostras com reposição, que convergem para uma distribuição bootstrap; a figura alerta que estrutura e vieses são herdados.](imagens/02_fluxo_bootstrap.svg)

Para cada repetição, sorteamos com reposição dentro de cada grupo e recalculamos
$\Delta_b=\bar{x}_{A,b}-\bar{x}_{B,b}$. A coleção dos valores aproxima a
variabilidade da estimativa sob o procedimento adotado.

In [ ]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd())) if str(Path.cwd()) not in sys.path else None
from graficos import histograma_referencia
from IPython.display import display

dados = pd.read_csv("dados/documentos.csv")
a = dados.loc[dados["local"].eq("Capital"), "palavras"].to_numpy()
b = dados.loc[dados["local"].eq("Interior"), "palavras"].to_numpy()
diferenca_observada = a.mean() - b.mean()
rng = np.random.default_rng(20260906)
B = 4000
diferencas_bootstrap = np.array([
    rng.choice(a, size=len(a), replace=True).mean()
    - rng.choice(b, size=len(b), replace=True).mean()
    for _ in range(B)
])
pd.Series(diferencas_bootstrap).describe(percentiles=[0.025, 0.5, 0.975])

A distribuição não mostra 4.000 novos passados possíveis: mostra a variação
produzida pela regra de reamostragem sobre os dados observados.

## 3. Intervalo de confiança

Usaremos o intervalo percentil introdutório:

$$
IC_{95\%}=\left[Q_{0{,}025}(\Delta_b),Q_{0{,}975}(\Delta_b)\right].
$$

Em repetições do procedimento, cerca de 95% dos intervalos construídos desta
maneira cobririam o parâmetro-alvo sob as condições do método. Depois de
calculado, não dizemos que há 95% de probabilidade frequentista de o parâmetro
fixo estar neste intervalo. O intervalo tampouco contém 95% dos documentos.

In [ ]:
limite_inferior, limite_superior = np.quantile(diferencas_bootstrap, [0.025, 0.975])
tabela_intervalo = pd.DataFrame([{
    "estimativa": diferenca_observada,
    "limite inferior": limite_inferior,
    "limite superior": limite_superior,
    "unidade": "palavras por documento",
}])
display(tabela_intervalo.round(2))
histograma_referencia(diferencas_bootstrap, diferenca_observada, "Distribuição bootstrap da diferença")

O intervalo apresenta valores compatíveis com o procedimento de estimação. Um
teste responde a outra pergunta: quão incompatível é o resultado com um modelo
nulo especificado?

## 4. Hipótese nula e teste de permutação

A hipótese nula didática afirma que, sob permutação, os rótulos `Capital` e
`Interior` são intercambiáveis. Mantemos valores e tamanhos dos grupos,
embaralhamos os rótulos e recalculamos a diferença.

![Histograma de uma distribuição nula destaca as duas caudas e uma linha para o resultado observado, sem representar um limiar automático.](imagens/02_distribuicao_nula.svg)

Para $M$ permutações, uma estimativa bilateral com correção finita é:

$$
p=\frac{1+\sum_{m=1}^{M}\mathbf{1}(|T_m|\geq|T_{obs}|)}{M+1}.
$$

In [ ]:
rng = np.random.default_rng(20260906)
combinados = np.concatenate([a, b])
M = 5000
permutadas = []
for _ in range(M):
    embaralhados = rng.permutation(combinados)
    permutadas.append(embaralhados[:len(a)].mean() - embaralhados[len(a):].mean())
diferencas_nulas = np.array(permutadas)
valor_p = (1 + np.sum(np.abs(diferencas_nulas) >= abs(diferenca_observada))) / (M + 1)
display(pd.Series({"diferença observada": diferenca_observada, "valor de p bilateral": valor_p}).round(4))
histograma_referencia(diferencas_nulas, diferenca_observada, "Diferenças sob permutação")

## 5. Como interpretar o valor de *p*

![Estimativa, intervalo, valor de p e contexto aparecem em quatro caixas distintas; nenhuma substitui desenho, transparência, teoria ou leitura dos casos.](imagens/02_mapa_interpretacao.svg)

O valor de *p* indica quão extremos seriam resultados como o observado sob a
hipótese nula e o procedimento especificado. Ele **não** informa:

- a probabilidade de a hipótese nula ser verdadeira;
- a importância do resultado;
- a probabilidade de replicação;
- a ausência de viés;
- a qualidade da operacionalização.

Conclusões não devem depender apenas de atravessar `0,05`. Relate estimativa,
intervalo, medida de efeito, desenho, decisões analíticas e contexto. A ASA
(Wasserstein e Lazar, 2016) fundamenta essas cautelas.

## 6. Significância e relevância substantiva

Uma diferença muito pequena pode produzir valor de *p* baixo em grande amostra;
uma diferença importante pode permanecer imprecisa em amostra pequena. A
relevância depende da pergunta, da escala, das consequências e da literatura.

**Atividade integrada — ficha inferencial. Modalidade:** dupla. **Tempo:** 30
minutos.

1. declare população-alvo e procedimento de seleção;
2. informe estimativa, intervalo e tamanho de efeito;
3. formule a hipótese nula do teste;
4. interprete o valor de *p* sem decisão binária;
5. indique duas fontes de incerteza não representadas;
6. escreva uma conclusão substantiva limitada.

**Minha ficha:** Escreva aqui.

**Situação em que eu não aplicaria inferência amostral:** Escreva aqui.

O eixo quantitativo está completo. No Notebook 03, a comparação muda de objeto:
representaremos documentos como termos e pesos antes de calcular semelhanças.